<a href="https://colab.research.google.com/github/bodduamarnath2004/Edge-ai-traffic-intelligent-system/blob/main/QuantisationAndPruning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q ultralytics pycocotools onnx onnxsim onnxruntime
!pip install -q torch-pruning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 5.5 MB/s eta 0:00:00


In [ ]:
import os, sys, json, shutil, time, warnings, zipfile
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch
import os, shutil, zipfile
from pathlib import Path
from pycocotools.coco import COCO
from ultralytics import YOLO
import torch_pruning as tp
warnings.filterwarnings("ignore")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
print("CUDA :", torch.cuda.is_available())
print("GPU  :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

CUDA : True
GPU  : Tesla T4


In [ ]:
class CFG:
    ROOT        = Path("/content")
    COCO_DIR    = ROOT / "coco_raw"
    DATA_DIR    = ROOT / "coco_vehicles"
    YAML        = DATA_DIR / "vehicles.yaml"
    PROJECT     = ROOT / "runs"
    COCO2YOLO   = {2: 0, 3: 1, 4: 2, 6: 3, 8: 4}
    NAMES       = ["bicycle", "car", "motorcycle", "bus", "truck"]
    NC          = 5
    MODEL       = "yolo11n.pt"
    IMGSZ       = 640
    BATCH           = 16
    WORKERS         = 2
    DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
    TRAIN_EPOCHS    = 50
    PRUNE_RATIO     = 0.35
    FINETUNE_EPOCHS = 50
    QAT_EPOCHS      = 50
    EXPORT_STEM = "yolo11n_vehicle_int8"

cfg = CFG()
cfg.PROJECT.mkdir(parents=True, exist_ok=True)
cfg.COCO_DIR.mkdir(parents=True, exist_ok=True)

def wget(url: str, dest: Path, label: str = ""):
    if dest.exists():
        print(f"cached  {dest.name}  ({dest.stat().st_size/1e6:.0f} MB)")
        return
    print(f"{label or dest.name}")
    os.system(f'wget -q --show-progress -O "{dest}" "{url}"')
    print(f"saved   {dest.name}  ({dest.stat().st_size/1e6:.0f} MB)")

def unzip(src: Path, dst: Path):
    print(f"extracting {src.name} …")
    with zipfile.ZipFile(src, "r") as z:
        z.extractall(dst)
    print(f"done")

def coco_to_yolo(split_src: str, img_dir: Path, ann_dir: Path, out_img: Path, out_lbl: Path):

    coco = COCO(str(ann_dir / f"instances_{split_src}.json"))
    img_ids = set()
    for cid in cfg.COCO2YOLO:
        img_ids.update(coco.getImgIds(catIds=[cid]))
    img_ids = sorted(img_ids)
    print(f"vehicle images in {split_src}: {len(img_ids):,}")
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)
    ok = skip = 0
    for info in coco.loadImgs(img_ids):
        src = img_dir / info["file_name"]
        if not src.exists():
            skip += 1
            continue
        ann_ids = coco.getAnnIds(imgIds=info["id"],catIds=list(cfg.COCO2YOLO),iscrowd=False)
        anns = coco.loadAnns(ann_ids)
        if not anns:
            skip += 1
            continue
        dst = out_img / info["file_name"]
        if not dst.exists():
            shutil.copy2(src, dst)
        W, H = info["width"], info["height"]
        lines = []
        for ann in anns:
            cls = cfg.COCO2YOLO.get(ann["category_id"])
            if cls is None:
                continue
            x, y, w, h = ann["bbox"]
            cx = max(0.0, min(1.0, (x + w / 2) / W))
            cy = max(0.0, min(1.0, (y + h / 2) / H))
            nw = max(0.0, min(1.0, w / W))
            nh = max(0.0, min(1.0, h / H))
            if nw > 1e-4 and nh > 1e-4:
                lines.append(f"{cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
        if lines:
            (out_lbl / (Path(info["file_name"]).stem + ".txt")).write_text("\n".join(lines))
            ok += 1
        else:
            skip += 1
    print(f"converted: {ok:,} skipped: {skip:,}")
    return ok

def build_valonly_dataset():
    ann_dir = cfg.COCO_DIR / "annotations"
    ann_zip = cfg.COCO_DIR / "annotations_trainval2017.zip"
    wget(
        "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
        ann_zip, "COCO annotations (180 MB)"
    )
    if not (ann_dir / "instances_val2017.json").exists():
        unzip(ann_zip, cfg.COCO_DIR)
    val_zip = cfg.COCO_DIR / "val2017.zip"
    val_dir = cfg.COCO_DIR / "val2017"
    wget(
        "http://images.cocodataset.org/zips/val2017.zip",
        val_zip, "val2017 images (1 GB)"
    )
    if not val_dir.exists():
        unzip(val_zip, cfg.COCO_DIR)
    tmp_img = cfg.DATA_DIR / "_tmp" / "images"
    tmp_lbl = cfg.DATA_DIR / "_tmp" / "labels"
    coco_to_yolo("val2017", val_dir, ann_dir, tmp_img, tmp_lbl)
    all_imgs = sorted(tmp_img.iterdir())
    rng      = np.random.default_rng(42)
    idx      = np.arange(len(all_imgs))
    rng.shuffle(idx)
    split_at = int(len(idx) * 0.80)

    splits = {
        "train2017": idx[:split_at],
        "val2017"  : idx[split_at:],
    }

    for split_name, idxs in splits.items():
        s_img = cfg.DATA_DIR / "images" / split_name
        s_lbl = cfg.DATA_DIR / "labels" / split_name
        s_img.mkdir(parents=True, exist_ok=True)
        s_lbl.mkdir(parents=True, exist_ok=True)
        for i in idxs:
            p = all_imgs[i]
            lbl = tmp_lbl / (p.stem + ".txt")
            shutil.copy2(p, s_img / p.name)
            if lbl.exists():
                shutil.copy2(lbl, s_lbl / lbl.name)
        print(f"  {split_name}: {len(idxs):,} images")

    shutil.rmtree(cfg.DATA_DIR / "_tmp", ignore_errors=True)
    yaml_txt = (
        f"path: {cfg.DATA_DIR}\n"
        f"train: images/train2017\n"
        f"val:   images/val2017\n\n"
        f"nc: {cfg.NC}\n"
        f"names: {cfg.NAMES}\n"
    )
    cfg.YAML.write_text(yaml_txt)
    print(f"\n✓ Dataset YAML → {cfg.YAML}")
    print(yaml_txt)

build_valonly_dataset()

COCO annotations (180 MB)
saved   annotations_trainval2017.zip  (253 MB)
extracting annotations_trainval2017.zip …
done
val2017 images (1 GB)
saved   val2017.zip  (816 MB)
extracting val2017.zip …
done
loading annotations into memory...
Done (t=0.51s)
creating index...
index created!
vehicle images in val2017: 870
converted: 870 skipped: 0
  train2017: 696 images
  val2017: 174 images

✓ Dataset YAML → /content/coco_vehicles/vehicles.yaml
path: /content/coco_vehicles
train: images/train2017
val:   images/val2017

nc: 5
names: ['bicycle', 'car', 'motorcycle', 'bus', 'truck']



In [ ]:
def train_baseline() -> Path:
    print("STEP 1 — Baseline Training  (YOLOv11n)")
    model = YOLO(cfg.MODEL)
    model.train(
        data          = str(cfg.YAML),
        epochs        = cfg.TRAIN_EPOCHS,
        imgsz         = cfg.IMGSZ,
        batch         = cfg.BATCH,
        device        = cfg.DEVICE,
        project       = str(cfg.PROJECT),
        name          = "1_baseline",
        exist_ok      = True,
        optimizer     = "AdamW",
        lr0           = 1e-3,
        lrf           = 0.01,
        momentum      = 0.937,
        weight_decay  = 5e-4,
        warmup_epochs = 3,
        cos_lr        = True,
        mosaic        = 1.0,
        mixup         = 0.1,
        copy_paste    = 0.1,
        hsv_h         = 0.015,
        hsv_s         = 0.7,
        hsv_v         = 0.4,
        degrees       = 5.0,
        translate     = 0.1,
        scale         = 0.5,
        fliplr        = 0.5,
        amp           = True,
        cache         = "ram",
        workers       = cfg.WORKERS,
        patience      = 20,
        verbose       = True,
    )
    path = cfg.PROJECT / "1_baseline" / "weights" / "best.pt"
    print(f"\Baseline best → {path}")
    return path
baseline_path = train_baseline()

STEP 1 — Baseline Training  (YOLOv11n)
Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/coco_vehicles/vehicles.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=1_baseline, nbs=64, nms=False, opset=None, optimize=False, opt

In [ ]:
def apply_pruning(model_path: Path, ratio: float = cfg.PRUNE_RATIO) -> Path:
    # if not HAS_TP:
    #     print("Skipping pruning step.")
    #     return model_path
    print(f"STEP 2 — Structured Channel Pruning  (ratio={ratio:.0%})")

    yolo = YOLO(str(model_path))
    pt   = yolo.model.float().to(cfg.DEVICE)
    pt.eval()
    dummy = torch.zeros(1, 3, cfg.IMGSZ, cfg.IMGSZ, device=cfg.DEVICE)
    from ultralytics.nn.modules.head import Detect
    ignored = []
    for m in pt.modules():
        if isinstance(m, Detect):
            ignored += list(m.modules())
    importance = tp.importance.MagnitudeImportance(p=2, group_reduction="mean")
    pruner = tp.pruner.MagnitudePruner(
        model          = pt,
        example_inputs = dummy,
        importance     = importance,
        pruning_ratio  = ratio,
        ignored_layers = ignored,
        global_pruning = True,
        round_to       = 8,
    )

    p0, _ = tp.utils.count_ops_and_params(pt, dummy)
    n0    = sum(x.numel() for x in pt.parameters())
    pruner.step()

    p1, _ = tp.utils.count_ops_and_params(pt, dummy)
    n1    = sum(x.numel() for x in pt.parameters())

    print(f"Params : {n0/1e6:.3f}M -> {n1/1e6:.3f}M  "
          f"({100*(1-n1/n0):.1f}%)")
    print(f"MACs   : {p0/1e9:.3f}G -> {p1/1e9:.3f}G  "
          f"({100*(1-p1/p0):.1f}%)")
    save_path = cfg.PROJECT / "pruned_model.pt"
    yolo.model = pt
    yolo.save(str(save_path))
    print(f"  ✓ Pruned model → {save_path}")
    return save_path

pruned_path = apply_pruning(baseline_path)


def finetune_pruned(model_path: Path, epochs: int = cfg.FINETUNE_EPOCHS) -> Path:
    print(f"STEP 3 — Fine-tune Pruned Model  ({epochs} epochs)")
    model = YOLO(str(model_path))
    model.train(data = str(cfg.YAML),
        epochs = epochs,
        imgsz = cfg.IMGSZ,
        batch = cfg.BATCH,
        device = cfg.DEVICE,
        project = str(cfg.PROJECT),
        name = "2_pruned_finetuned",
        exist_ok = True,
        optimizer = "AdamW",
        lr0 = 1e-4,
        lrf = 0.01,
        cos_lr = True,
        warmup_epochs = 1,
        amp = True,
        cache = "ram",
        workers = cfg.WORKERS,
        patience = 15,
        verbose = True,)

    path = cfg.PROJECT / "2_pruned_finetuned" / "weights" / "best.pt"
    print(f"\nFine-tuned pruned model → {path}")
    return path

finetuned_path = finetune_pruned(pruned_path)

STEP 2 — Structured Channel Pruning  (ratio=35%)
Params : 2.591M -> 2.591M  (0.0%)
MACs   : 3.191G -> 3.191G  (0.0%)
  ✓ Pruned model → /content/runs/pruned_model.pt
STEP 3 — Fine-tune Pruned Model  (50 epochs)
Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/coco_vehicles/vehicles.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ra

In [ ]:
from torch.ao.quantization import (prepare_qat,QConfig,)
from torch.ao.quantization.observer import (MovingAverageMinMaxObserver,MovingAveragePerChannelMinMaxObserver,)
from torch.ao.quantization.fake_quantize import FakeQuantize

def _qat_qconfig() -> QConfig:
    act_fq = FakeQuantize.with_args(
        observer     = MovingAverageMinMaxObserver,
        quant_min    = 0,
        quant_max    = 255,
        dtype        = torch.quint8,
        reduce_range = False,
    )
    wt_fq = FakeQuantize.with_args(
        observer     = MovingAveragePerChannelMinMaxObserver,
        quant_min    = -128,
        quant_max    = 127,
        dtype        = torch.qint8,
        ch_axis      = 0,
        reduce_range = False,
    )
    return QConfig(activation=act_fq, weight=wt_fq)


def inject_qat(pt_model: nn.Module) -> nn.Module:
    try:
        from ultralytics.nn.modules.head import Detect
        protected = {id(m) for m in pt_model.modules() if isinstance(m, Detect)}
    except ImportError:
        protected = set()

    qcfg = _qat_qconfig()
    count = 0
    for m in pt_model.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)) and id(m) not in protected:
            m.qconfig = qcfg
            count += 1
    print(f"QAT qconfig attached to {count} layers")
    return pt_model


def run_qat(model_path: Path, epochs: int = cfg.QAT_EPOCHS) -> Path:
    print(f"STEP 4 — Quantization-Aware Training  ({epochs} epochs)")
    yolo = YOLO(str(model_path))
    pt   = yolo.model.float().cpu()
    inject_qat(pt)
    pt_qat = prepare_qat(pt.train())
    pt_qat = pt_qat.to(cfg.DEVICE)
    yolo.model = pt_qat
    print("FakeQuantize nodes inserted")

    yolo.train(
        data          = str(cfg.YAML),
        epochs        = epochs,
        imgsz         = cfg.IMGSZ,
        batch         = max(8, cfg.BATCH // 2),
        device        = cfg.DEVICE,
        project       = str(cfg.PROJECT),
        name          = "3_qat",
        exist_ok      = True,
        optimizer     = "AdamW",
        lr0           = 5e-5,
        lrf           = 0.01,
        cos_lr        = True,
        warmup_epochs = 0,
        amp           = False,
        cache         = "ram",
        workers       = cfg.WORKERS,
        patience      = epochs,
        verbose       = True,
    )

    path = cfg.PROJECT / "3_qat" / "weights" / "best.pt"
    print(f"\nQAT model → {path}")
    return path

qat_path = run_qat(finetuned_path)

STEP 4 — Quantization-Aware Training  (50 epochs)
QAT qconfig attached to 88 layers
FakeQuantize nodes inserted
Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/coco_vehicles/vehicles.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=5e-05, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/runs/2_pruned_finetuned/weights/best.pt, m

In [ ]:
def validate(model_path: Path, tag: str) -> dict:
    yolo    = YOLO(str(model_path))
    metrics = yolo.val(
        data    = str(cfg.YAML),
        imgsz   = cfg.IMGSZ,
        batch   = cfg.BATCH,
        device  = cfg.DEVICE,
        workers = cfg.WORKERS,
        verbose = False,
    )
    r = {"mAP50": metrics.box.map50, "mAP50-95": metrics.box.map}
    print(f"  {tag:42s}  mAP50={r['mAP50']:.4f}  mAP50-95={r['mAP50-95']:.4f}")
    return r

print("VALIDATION SUMMARY")
r_base = validate(baseline_path,  "Baseline   FP32")
r_ft   = validate(finetuned_path, "Pruned + Fine-tuned   FP32")
r_qat  = validate(qat_path,       "QAT   FP32 (fake-quant active)")

drop = r_base["mAP50"] - r_qat["mAP50"]
print(f"\n  mAP50 drop baseline -> QAT : {drop:+.4f}")
if drop > 0.03:
    print(" Drop > 3 pp — consider raising QAT_EPOCHS or lowering PRUNE_RATIO")
else:
    print(" Acceptable accuracy drop (< 3 pp)")

VALIDATION SUMMARY
Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,583,127 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3288.0±587.8 MB/s, size: 196.8 KB)
val: Scanning /content/coco_vehicles/labels/val2017.cache... 174 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 174/174 60.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 3.2it/s 3.5s
                   all        174        639      0.568       0.44      0.453      0.271
Speed: 2.8ms preprocess, 5.6ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to /content/runs/detect/val
  Baseline   FP32                             mAP50=0.4529  mAP50-95=0.2713
Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,583,127 parameters, 0 gradients, 6.3 GFLOPs
val: Fas

In [ ]:
from ultralytics import YOLO
from pathlib import Path

print("STEP 5 — Export FP32 ONNX (then quantize to INT8)")
model = YOLO(str(qat_path))
onnx_path = model.export(
    format="onnx",
    imgsz=cfg.IMGSZ,
    simplify=True,
    opset=13,
)

print(f"\nFP32 ONNX exported → {onnx_path}")

print("\nConverting to INT8")

from onnxruntime.quantization import quantize_dynamic, QuantType

int8_path = str(Path(onnx_path).with_suffix(".int8.onnx"))

quantize_dynamic(
    model_input=onnx_path,
    model_output=int8_path,
    weight_type=QuantType.QInt8
)

print(f"INT8 model saved → {int8_path}")

final_path = Path(cfg.PROJECT) / f"{cfg.EXPORT_STEM}.onnx"
Path(int8_path).rename(final_path)

print(f"Final model → {final_path}")
size_mb = final_path.stat().st_size / 1e6
print(f"📦 Final INT8 model size: {size_mb:.2f} MB")

STEP 5 — Export FP32 ONNX (then quantize to INT8)
Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11n summary (fused): 101 layers, 2,583,127 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from '/content/runs/3_qat/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 9, 8400) (5.2 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 444ms
Prepared 3 packages in 5.59s
Installed 3 packages in 39ms
 + colorama==0.4.6
 + onnxruntime-gpu==1.25.1
 + onnxslim==0.1.92

requirements: AutoUpdate success ✅ 6.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 13...
ONNX: 

INT8 model saved → /content/runs/3_qat/weights/best.int8.onnx
Final model → /content/runs/yolo11n_vehicle_int8.onnx
📦 Final INT8 model size: 3.01 MB


In [ ]:
from pathlib import Path
import shutil

print("Saving final QAT model (.pt)")

src = Path(qat_path)

dst = Path(cfg.PROJECT) / "yolo11n_vehicle_qat_final.pt"

shutil.copy(src, dst)

print(f"Final model saved → {dst}")
print(f"Size: {dst.stat().st_size / 1e6:.2f} MB")

Saving final QAT model (.pt)
Final model saved → /content/runs/yolo11n_vehicle_qat_final.pt
Size: 5.45 MB


In [ ]:
import torch
import time
import pandas as pd
from ultralytics import YOLO

try:
    from thop import profile
    HAS_THOP = True
except:
    HAS_THOP = False
    print("THOP not installed → FLOPs will be skipped")

models_dict = {"Baseline": baseline_path,"Pruned+FT": finetuned_path,"QAT": qat_path,}
results_table = []
device = cfg.DEVICE
imgsz = cfg.IMGSZ

def measure_latency(model, runs=50):
    dummy = torch.randn(1, 3, imgsz, imgsz).to(device)
    for _ in range(10):
        _ = model(dummy)
    if device == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    for _ in range(runs):
        _ = model(dummy)
    if device == "cuda":
        torch.cuda.synchronize()
    end = time.time()
    return (end - start) / runs


for name, path in models_dict.items():
    print(f"\nProcessing: {name}")
    yolo = YOLO(str(path))
    model = yolo.model.to(device).eval()
    params = sum(p.numel() for p in model.parameters()) / 1e6
    if HAS_THOP:
        dummy = torch.randn(1, 3, imgsz, imgsz).to(device)
        flops, _ = profile(model, inputs=(dummy,), verbose=False)
        flops = flops / 1e9
    else:
        flops = -1
    latency = measure_latency(model)
    fps = 1.0 / latency
    size_mb = path.stat().st_size / 1e6
    memory_mb = params * 4
    metrics = yolo.val(
        data=str(cfg.YAML),
        imgsz=imgsz,
        batch=cfg.BATCH,
        device=device,
        verbose=False,)
    map50 = metrics.box.map50
    map5095 = metrics.box.map

    results_table.append({
        "Model": name,
        "Params (M)": round(params, 3),
        "GFLOPs": round(flops, 3) if flops != -1 else "N/A",
        "Latency (ms)": round(latency * 1000, 2),
        "FPS": round(fps, 2),
        "Model Size (MB)": round(size_mb, 2),
        "Est. RAM (MB)": round(memory_mb, 2),
        "mAP50": round(map50, 4),
        "mAP50-95": round(map5095, 4),
    })

df = pd.DataFrame(results_table)
df = df.sort_values(by="Latency (ms)")
print("MODEL COMPARISON")

display(df)


Processing: Baseline
Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,583,127 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2418.9±632.6 MB/s, size: 173.8 KB)
val: Scanning /content/coco_vehicles/labels/val2017.cache... 174 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 174/174 48.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 2.9it/s 3.8s
                   all        174        639      0.568       0.44      0.453      0.271
Speed: 4.0ms preprocess, 3.9ms inference, 0.0ms loss, 2.4ms postprocess per image
Results saved to /content/runs/detect/val-4

Processing: Pruned+FT
Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,583,127 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3133.5

,Model,Params (M),GFLOPs,Latency (ms),FPS,Model Size (MB),Est. RAM (MB),mAP50,mAP50-95
2,QAT,2.591,3.222,11.29,88.56,5.45,10.36,0.4585,0.2862
0,Baseline,2.591,3.222,11.37,87.95,5.45,10.36,0.4529,0.2713
1,Pruned+FT,2.591,3.222,11.55,86.58,5.45,10.36,0.4536,0.2889


In [ ]:
from ultralytics import YOLO
from pathlib import Path

print("Running QAT model on video")
model = YOLO(str(qat_path))
video_path = "my_video.mp4"
output_path = str(Path(cfg.PROJECT) / "qat_output_fixed.mp4")
print(f"Input  : {video_path}")
print(f"Output : {output_path}")

results = model.predict(
    source=video_path,
    imgsz=cfg.IMGSZ,
    conf=0.45,
    iou=0.40,
    tracker="botsort.yaml",
    save=True,
    project=str(cfg.PROJECT),
    name="qat_output_fixed",
    exist_ok=True,
    verbose=False,
    max_det=50
)

print(f"\nVideo successfully saved → {results[0].save_dir}/qat_output_fixed.mp4")

Running QAT model on video
Input  : my_video.mp4
Output : /content/runs/qat_output_fixed.mp4
WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

WARNING ⚠️ NMS time limit 2.050s exceeded
